# Aegis AML Training Notebook for Colab Pro

This notebook is meant for a cold Colab Pro runtime.

It downloads the Elliptic dataset directly from Kaggle, stages the expected CSVs into the repo, and then runs the repository training commands individually so the final output layout matches `models/`.

## Repo and dataset source

Use one of these repo modes:
- `REPO_MODE = "git"`: clone from GitHub into `/content`
- `REPO_MODE = "drive"`: use a repo folder already stored in Google Drive

Kaggle auth modes:
- `KAGGLE_AUTH_MODE = "upload"`: upload `kaggle.json` directly in Colab
- `KAGGLE_AUTH_MODE = "drive"`: read `kaggle.json` from Google Drive

Kaggle dataset slug: `ellipticco/elliptic-data-set`

Dataset page: https://www.kaggle.com/datasets/ellipticco/elliptic-data-set

In [ ]:
from __future__ import annotations

# Repo location
REPO_MODE = "git"  # "git" or "drive"
REPO_URL = ""      # Example: https://github.com/<user>/Aegis-AML.git
REPO_BRANCH = "main"
DRIVE_REPO_PATH = "/content/drive/MyDrive/Aegis-AML"

# Kaggle auth
KAGGLE_AUTH_MODE = "upload"  # "upload" or "drive"
KAGGLE_JSON_PATH = "/content/drive/MyDrive/kaggle/kaggle.json"
KAGGLE_DATASET = "ellipticco/elliptic-data-set"

# Data + outputs
DATA_DIR_NAME = "processed"
RESET_OUTPUTS = True

# Output handling
EXPORT_ARTIFACTS_TO_DRIVE = True
DRIVE_EXPORT_DIR = "/content/drive/MyDrive/Aegis-AML-colab-outputs"


In [ ]:
from pathlib import Path
import os
import shutil
import sys

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    raise RuntimeError("This notebook is intended for Google Colab.")

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import subprocess

WORKSPACE_ROOT = Path("/content")

if REPO_MODE == "git":
    if not REPO_URL.strip():
        raise ValueError("Set REPO_URL before running the notebook in git mode.")
    REPO_ROOT = WORKSPACE_ROOT / "Aegis-AML"
    if REPO_ROOT.exists():
        print(f"Using existing clone at {REPO_ROOT}")
    else:
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
elif REPO_MODE == "drive":
    REPO_ROOT = Path(DRIVE_REPO_PATH).resolve()
    if not REPO_ROOT.exists():
        raise FileNotFoundError(f"Drive repo path not found: {REPO_ROOT}")
else:
    raise ValueError("REPO_MODE must be 'git' or 'drive'")

BACKEND_DIR = REPO_ROOT / "backend"
DATA_ROOT = REPO_ROOT / "data"
DATA_EXTERNAL = DATA_ROOT / "external"
DATA_DIR = DATA_ROOT / DATA_DIR_NAME
MODELS_DIR = REPO_ROOT / "models"
KAGGLE_DOWNLOAD_DIR = WORKSPACE_ROOT / "kaggle_download"

if not BACKEND_DIR.exists():
    raise FileNotFoundError(f"Backend directory not found: {BACKEND_DIR}")

os.chdir(REPO_ROOT)
sys.path.insert(0, str(BACKEND_DIR))

print(f"Repo root: {REPO_ROOT}")
print(f"Backend dir: {BACKEND_DIR}")
print(f"Input data dir: {DATA_EXTERNAL}")
print(f"Processed data dir: {DATA_DIR}")
print(f"Models dir: {MODELS_DIR}")

In [ ]:
def pip_install(args: list[str]) -> None:
    cmd = [sys.executable, "-m", "pip", "install"] + args
    print("+", " ".join(cmd))
    subprocess.run(cmd, check=True)


pip_install(["--upgrade", "pip"])

pip_install([
    "kaggle>=1.6.17",
    "pandas>=2.2.0",
    "numpy>=1.26.0",
    "scikit-learn>=1.5.0",
    "xgboost>=2.1.0",
    "lightgbm>=4.5.0",
    "torch-geometric>=2.6.0",
    "networkx>=3.3",
    "python-louvain>=0.16",
    "leidenalg>=0.10.0",
    "igraph>=0.11.0",
    "shap>=0.46.0",
    "pydantic>=2.9.0",
    "pydantic-settings>=2.6.0",
    "python-multipart>=0.0.12",
    "python-dotenv>=1.0.0",
    "joblib>=1.4.0",
    "optuna>=4.1.0",
    "httpx>=0.27.0",
    "PyJWT>=2.8.0",
    "cryptography>=42.0.0",
    "reportlab>=4.0.0",
    "PyPDF2>=3.0.0",
    "Pillow>=10.0.0",
])

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
target_json = kaggle_dir / "kaggle.json"

if KAGGLE_AUTH_MODE == "upload":
    from google.colab import files
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise FileNotFoundError("Upload a file named kaggle.json to continue.")
    target_json.write_bytes(uploaded["kaggle.json"])
elif KAGGLE_AUTH_MODE == "drive":
    kaggle_json = Path(KAGGLE_JSON_PATH).resolve()
    if not kaggle_json.exists():
        raise FileNotFoundError(
            f"Kaggle credentials not found at {kaggle_json}. Update KAGGLE_JSON_PATH or switch KAGGLE_AUTH_MODE to upload."
        )
    shutil.copy2(kaggle_json, target_json)
else:
    raise ValueError("KAGGLE_AUTH_MODE must be 'upload' or 'drive'")

target_json.chmod(0o600)
print(f"Kaggle auth ready at {target_json}")

## Download Elliptic directly from Kaggle

In [ ]:
if KAGGLE_DOWNLOAD_DIR.exists():
    shutil.rmtree(KAGGLE_DOWNLOAD_DIR)
KAGGLE_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run([
    sys.executable,
    "-m",
    "kaggle",
    "datasets",
    "download",
    "-d",
    KAGGLE_DATASET,
    "-p",
    str(KAGGLE_DOWNLOAD_DIR),
    "--unzip",
], check=True)

print("Downloaded to", KAGGLE_DOWNLOAD_DIR)
for path in sorted(KAGGLE_DOWNLOAD_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(KAGGLE_DOWNLOAD_DIR))

In [ ]:
if RESET_OUTPUTS:
    for path in [DATA_EXTERNAL, DATA_DIR, MODELS_DIR]:
        if path.exists():
            shutil.rmtree(path)

DATA_EXTERNAL.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

for subdir in [
    MODELS_DIR / "behavioral",
    MODELS_DIR / "graph",
    MODELS_DIR / "entity",
    MODELS_DIR / "temporal",
    MODELS_DIR / "offramp",
    MODELS_DIR / "meta",
    MODELS_DIR / "artifacts",
]:
    subdir.mkdir(parents=True, exist_ok=True)

print("Output directories prepared")

In [ ]:
csv_map = {csv_path.name: csv_path for csv_path in KAGGLE_DOWNLOAD_DIR.rglob("*.csv")}
required_files = [
    "elliptic_txs_features.csv",
    "elliptic_txs_classes.csv",
    "elliptic_txs_edgelist.csv",
]

missing = [name for name in required_files if name not in csv_map]
if missing:
    raise FileNotFoundError(f"Downloaded dataset is missing expected CSVs: {missing}")

for name in required_files:
    shutil.copy2(csv_map[name], DATA_EXTERNAL / name)
    print(f"Copied {name} -> {DATA_EXTERNAL / name}")


def run(cmd: list[str], cwd: Path = BACKEND_DIR) -> None:
    print("+", " ".join(str(part) for part in cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


expected_inputs = [
    DATA_EXTERNAL / "elliptic_txs_features.csv",
    DATA_EXTERNAL / "elliptic_txs_classes.csv",
    DATA_EXTERNAL / "elliptic_txs_edgelist.csv",
]
for path in expected_inputs:
    print(f"{'OK' if path.exists() else 'MISSING'}  {path}")

## Dataset check

In [ ]:
expected_inputs = [
    DATA_EXTERNAL / "elliptic_txs_features.csv",
    DATA_EXTERNAL / "elliptic_txs_classes.csv",
    DATA_EXTERNAL / "elliptic_txs_edgelist.csv",
]

for path in expected_inputs:
    print(f"{'OK' if path.exists() else 'MISSING'}  {path}")

## 1. Optional feature preparation

In [ ]:
if not SKIP_FEATURES:
    prepare_features(input_dir=DATA_EXTERNAL, data_dir=DATA_DIR)
else:
    print("Skipping feature preparation")

## 2. Train the lens models

In [ ]:
if RUN_PARALLEL_LENSES:
    train_parallel_lenses(data_dir=DATA_DIR)
else:
    train_sequential_lenses(data_dir=DATA_DIR)

## 3. Train the entity model

In [ ]:
train_entity(data_dir=DATA_DIR)

## 4. Score training data

In [ ]:
score_training_data(data_dir=DATA_DIR)

## 5. Build meta features

In [ ]:
prepare_meta_features(data_dir=DATA_DIR)

## 6. Train the meta learner

In [ ]:
train_meta(data_dir=DATA_DIR)

## One-cell full pipeline

In [ ]:
# train_all(skip_features=SKIP_FEATURES, input_dir=DATA_EXTERNAL, data_dir=DATA_DIR)

## Artifact check

In [ ]:
expected_artifacts = [
    MODELS_DIR / "behavioral" / "xgboost_behavioral.pkl",
    MODELS_DIR / "behavioral" / "autoencoder_behavioral.pt",
    MODELS_DIR / "graph" / "gat_model.pt",
    MODELS_DIR / "graph" / "node_embeddings.npy",
    MODELS_DIR / "entity" / "entity_classifier.pkl",
    MODELS_DIR / "temporal" / "lstm_model.pt",
    MODELS_DIR / "offramp" / "offramp_classifier.pkl",
    MODELS_DIR / "meta" / "meta_model.pkl",
    MODELS_DIR / "artifacts" / "threshold_config.json",
    MODELS_DIR / "artifacts" / "metrics_report.json",
    DATA_DIR / "train_features_scored.csv",
    DATA_DIR / "meta_features.csv",
]

for artifact in expected_artifacts:
    print(f"{'OK' if artifact.exists() else 'MISSING'}  {artifact}")

## Export artifacts to Drive

In [ ]:
if EXPORT_ARTIFACTS_TO_DRIVE:
    export_root = Path(DRIVE_EXPORT_DIR).resolve()
    export_root.mkdir(parents=True, exist_ok=True)

    model_export_dir = export_root / "models"
    data_export_dir = export_root / "data"

    if model_export_dir.exists():
        shutil.rmtree(model_export_dir)
    shutil.copytree(MODELS_DIR, model_export_dir)

    data_export_dir.mkdir(parents=True, exist_ok=True)
    for filename in ["train_features.csv", "train_features_scored.csv", "meta_features.csv", "edges.csv", "node_labels.csv", "wallet_labels.csv"]:
        src = DATA_DIR / filename
        if src.exists():
            shutil.copy2(src, data_export_dir / filename)

    print(f"Exported outputs to {export_root}")
else:
    print("Artifact export skipped")